In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import gc
import torch

# Delete any old model variables if they exist in the namespace
if 'model' in locals():
    del model
if 'optimizer' in locals():
    del optimizer

# Force garbage collection and empty the PyTorch cache
gc.collect()
torch.cuda.empty_cache()

In [3]:
import os, math
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    get_cosine_schedule_with_warmup,
)
import wandb

# ─────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────
DATA_DIR        = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
MODEL_NAME      = "microsoft/deberta-v3-base" 
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN         = 256        
BATCH_SIZE      = 2
ACCUM_STEPS     = 8           
N_FOLDS         = 5
EPOCHS          = 4
LR_BACKBONE     = 1e-5
LR_HEAD         = 1e-4
WARMUP_FRAC     = 0.1
GRAD_CLIP       = 1.0
LABEL_SMOOTHING = 0.15
HEAD_DROPOUT    = 0.2
GRAD_CKPT       = True        
USE_FP16        = DEVICE == "cuda"
OPTION_COLS     = ["A", "B", "C", "D", "E"]

# ─────────────────────────────────────────────────────────
# W&B 
# ─────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")
 
if not WANDB_API_KEY:
    raise ValueError(
        "WANDB_API_KEY not found. "
        "Go to Kaggle notebook -> Add-ons -> Secrets -> New Secret. "
        "Name: WANDB_API_KEY, Value: your key from https://wandb.ai/authorize"
    )
 
wandb.login(key=WANDB_API_KEY)
wandb.init(
    project="smart-mcq-solver",
    name=f"deberta-v3-base-{N_FOLDS}fold",
    config=dict(
        model=MODEL_NAME, max_len=MAX_LEN,
        batch_size=BATCH_SIZE, accum_steps=ACCUM_STEPS,
        effective_batch=BATCH_SIZE * ACCUM_STEPS,
        n_folds=N_FOLDS, epochs=EPOCHS,
        lr_backbone=LR_BACKBONE, lr_head=LR_HEAD,
        warmup_frac=WARMUP_FRAC, grad_clip=GRAD_CLIP,
        label_smoothing=LABEL_SMOOTHING, head_dropout=HEAD_DROPOUT,
        grad_ckpt=GRAD_CKPT, fp16=USE_FP16,
    ),
)

print(f"Device : {DEVICE}  |  FP16: {USE_FP16}  |  GradCkpt: {GRAD_CKPT}")
print(f"Model  : {MODEL_NAME}")
print(f"Folds  : {N_FOLDS}  |  Epochs/fold: {EPOCHS}")
print(f"MAX_LEN: {MAX_LEN}  |  Effective batch: {BATCH_SIZE * ACCUM_STEPS}")
print(f"W&B    : {wandb.run.get_url()}")


# ─────────────────────────────────────────────────────────
# MAP@3
# ─────────────────────────────────────────────────────────
def apk(actual, predicted, k=3):
    if not actual: return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted[:k]):
        if p in actual and p not in predicted[:i]:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(actual), k)

def mapk(actuals, predictions, k=3):
    return np.mean([apk([a], p, k) for a, p in zip(actuals, predictions)])


# ─────────────────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")
train_df = train_df.dropna(subset=["prompt"] + OPTION_COLS + ["answer"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["prompt"] + OPTION_COLS).reset_index(drop=True)
print(f"Train: {len(train_df)}  |  Test: {len(test_df)}")

label_map = {c: i for i, c in enumerate(OPTION_COLS)}
train_df["label_idx"] = train_df["answer"].map(label_map)


# ─────────────────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────────────────
class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, has_labels=True):
        self.df         = df.reset_index(drop=True)
        self.tok        = tokenizer
        self.has_labels = has_labels

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        q       = str(row["prompt"])
        choices = [str(row[c]) for c in OPTION_COLS]
        enc = self.tok(
            [q] * 5, choices,
            truncation=True, max_length=MAX_LEN,
            padding="max_length", return_tensors="pt",
        )
        item = {k: v for k, v in enc.items()}
        if self.has_labels:
            item["labels"] = torch.tensor(int(row["label_idx"]), dtype=torch.long)
        return item


# ─────────────────────────────────────────────────────────
# MODEL BUILDER
# ─────────────────────────────────────────────────────────
def build_model():
    model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME).to(DEVICE)
    model = model.float()

    # gradient checkpointing — trades compute for memory
    if GRAD_CKPT:
        model.deberta.encoder.gradient_checkpointing = True

    # dropout before classifier head
    if hasattr(model, "classifier"):
        model.classifier = nn.Sequential(
            nn.Dropout(HEAD_DROPOUT),
            model.classifier,
        )
    return model


def build_optimizer(model, total_steps):
    head_params = ["classifier", "pooler"]
    no_decay    = ["bias", "LayerNorm.weight"]
    groups = [
        {"params": [p for n, p in model.named_parameters()
                    if not any(nd in n for nd in no_decay) and not any(h in n for h in head_params)],
         "lr": LR_BACKBONE, "weight_decay": 0.01},
        {"params": [p for n, p in model.named_parameters()
                    if any(nd in n for nd in no_decay) and not any(h in n for h in head_params)],
         "lr": LR_BACKBONE, "weight_decay": 0.0},
        {"params": [p for n, p in model.named_parameters()
                    if not any(nd in n for nd in no_decay) and any(h in n for h in head_params)],
         "lr": LR_HEAD, "weight_decay": 0.01},
        {"params": [p for n, p in model.named_parameters()
                    if any(nd in n for nd in no_decay) and any(h in n for h in head_params)],
         "lr": LR_HEAD, "weight_decay": 0.0},
    ]
    optimizer = AdamW(groups, eps=1e-6)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_FRAC * total_steps),
        num_training_steps=total_steps,
    )
    return optimizer, scheduler


# ─────────────────────────────────────────────────────────
# TRAIN ONE FOLD
# ─────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

def train_fold(fold, train_idx, val_idx, tokenizer):
    print(f"\n{'='*55}\nFOLD {fold+1}/{N_FOLDS}\n{'='*55}")

    tr_dl = DataLoader(MCQDataset(train_df.iloc[train_idx], tokenizer),
                       batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    vl_dl = DataLoader(MCQDataset(train_df.iloc[val_idx],   tokenizer),
                       batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = build_model()
    total_steps = (len(tr_dl) // ACCUM_STEPS) * EPOCHS
    optimizer, scheduler = build_optimizer(model, total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)

    best_val_map3 = 0.0
    best_ckpt     = f"/kaggle/working/fold{fold}_best.pt"
    global_step   = 0

    for epoch in range(EPOCHS):
        # ── Train ──────────────────────────────────────────
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        optimizer.zero_grad()

        for step, batch in enumerate(tr_dl):
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbl  = batch["labels"].to(DEVICE)

            with torch.amp.autocast("cuda", enabled=USE_FP16):
                out  = model(input_ids=ids, attention_mask=mask)
                loss = criterion(out.logits, lbl) / ACCUM_STEPS

            if not math.isfinite(loss.item() * ACCUM_STEPS):
                optimizer.zero_grad(); continue

            scaler.scale(loss).backward()

            if (step + 1) % ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)      
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()         
                global_step += 1

                wandb.log({
                    f"fold{fold+1}/train/step_loss":   loss.item() * ACCUM_STEPS,
                    f"fold{fold+1}/train/lr_backbone": scheduler.get_last_lr()[0],
                    f"fold{fold+1}/train/lr_head":     scheduler.get_last_lr()[-1],
                }, step=global_step)

            total_loss += loss.item() * ACCUM_STEPS
            correct    += (out.logits.argmax(-1) == lbl).sum().item()
            total      += lbl.size(0)

        epoch_loss = total_loss / len(tr_dl)
        epoch_acc  = correct / max(total, 1)
        print(f"  Ep{epoch+1} — loss:{epoch_loss:.4f}  acc:{epoch_acc:.4f}")

        # ── Validate ───────────────────────────────────────
        model.eval()
        val_logits, val_labels = [], []
        with torch.no_grad():
            for batch in vl_dl:
                ids  = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                lbl  = batch["labels"]
                with torch.amp.autocast("cuda", enabled=USE_FP16):
                    out = model(input_ids=ids, attention_mask=mask)
                val_logits.append(out.logits.float().cpu().numpy())
                val_labels.extend(lbl.numpy().tolist())

        val_logits = np.vstack(val_logits)
        val_preds  = [[OPTION_COLS[i] for i in np.argsort(r)[::-1]][:3] for r in val_logits]
        val_map3   = mapk([OPTION_COLS[l] for l in val_labels], val_preds)
        print(f"         val MAP@3 = {val_map3:.4f}")

        wandb.log({
            f"fold{fold+1}/train/epoch_loss": epoch_loss,
            f"fold{fold+1}/train/epoch_acc":  epoch_acc,
            f"fold{fold+1}/val/map3":         val_map3,
            "epoch": epoch + 1,
            "fold":  fold + 1,
        }, step=global_step)

        if val_map3 > best_val_map3:
            best_val_map3 = val_map3
            torch.save(model.state_dict(), best_ckpt)
            wandb.run.summary[f"fold{fold+1}/best_val_map3"] = best_val_map3
            print(f"         ✓ saved (val MAP@3={best_val_map3:.4f})")

    # ── Test inference with best checkpoint ────────────────
    model.load_state_dict(torch.load(best_ckpt))
    model.eval()
    fold_logits = []
    with torch.no_grad():
        for batch in DataLoader(MCQDataset(test_df, tokenizer, has_labels=False),
                                batch_size=BATCH_SIZE, num_workers=2):
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            with torch.amp.autocast("cuda", enabled=USE_FP16):
                out = model(input_ids=ids, attention_mask=mask)
            fold_logits.append(out.logits.float().cpu().numpy())

    del model
    torch.cuda.empty_cache()
    return np.vstack(fold_logits), best_val_map3


# ─────────────────────────────────────────────────────────
# CV LOOP
# ─────────────────────────────────────────────────────────
print("\nLoading tokenizer …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
all_fold_logits, fold_scores = [], []

for fold, (tr_idx, vl_idx) in enumerate(skf.split(train_df, train_df["label_idx"])):
    logits, score = train_fold(fold, tr_idx, vl_idx, tokenizer)
    all_fold_logits.append(logits)
    fold_scores.append(score)

print(f"\n{'='*55}")
print(f"CV MAP@3 per fold: {[f'{s:.4f}' for s in fold_scores]}")
print(f"Mean CV MAP@3    : {np.mean(fold_scores):.4f}")
print(f"{'='*55}")

# W&B summary
wandb.run.summary["cv_map3_mean"] = float(np.mean(fold_scores))
wandb.run.summary["cv_map3_std"]  = float(np.std(fold_scores))
wandb.log({
    "cv_fold_map3": wandb.plot.bar(
        wandb.Table(columns=["fold", "val_map3"],
                    data=[[f"fold{i+1}", s] for i, s in enumerate(fold_scores)]),
        "fold", "val_map3", title="Val MAP@3 per Fold",
    )
})


# ─────────────────────────────────────────────────────────
# ENSEMBLE & SUBMIT
# ─────────────────────────────────────────────────────────
avg_logits = np.mean(all_fold_logits, axis=0)
test_preds = [[OPTION_COLS[i] for i in np.argsort(r)[::-1]][:3] for r in avg_logits]

submission = pd.DataFrame({
    "ID":         test_df["id"],
    "Prediction": [" ".join(p) for p in test_preds],
})
out_path = "/kaggle/working/submission_deberta_v4.csv"
submission.to_csv(out_path, index=False)
print(f"\n✅  Saved → {out_path}")
print(submission.head(10).to_string(index=False))

wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260610_042131-33x3fkyg
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run deberta-v3-base-5fold
wandb: ⭐️ View project at https://wandb.ai/uma-shettar-indian-institute-of-technology-madras/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/uma-shettar-indian-institute-of-technology-madras/smart-mcq-solver/runs/33x3fkyg
wandb: WARNING The get_url method is deprecated and wi

Device : cuda  |  FP16: True  |  GradCkpt: True
Model  : microsoft/deberta-v3-base
Folds  : 5  |  Epochs/fold: 4
MAX_LEN: 256  |  Effective batch: 16
W&B    : https://wandb.ai/uma-shettar-indian-institute-of-technology-madras/smart-mcq-solver/runs/33x3fkyg
Train: 2000  |  Test: 500

Loading tokenizer …


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

...

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                

  Ep1 — loss:1.5925  acc:0.2450
         val MAP@3 = 0.5825
         ✓ saved (val MAP@3=0.5825)
  Ep2 — loss:1.4067  acc:0.4306
         val MAP@3 = 0.8146
         ✓ saved (val MAP@3=0.8146)
  Ep3 — loss:1.2333  acc:0.5669
         val MAP@3 = 0.8629
         ✓ saved (val MAP@3=0.8629)
  Ep4 — loss:1.1643  acc:0.6138
         val MAP@3 = 0.8754
         ✓ saved (val MAP@3=0.8754)

CV MAP@3 per fold: ['0.8688', '0.8729', '0.8042', '0.8696', '0.8754']
Mean CV MAP@3    : 0.8582


wandb: updating run metadata; uploading artifact run-33x3fkyg-cv_fold_map3_table



✅  Saved → /kaggle/working/submission_deberta_v4.csv
 ID Prediction
  1      A D B
  2      B D E
  3      B E D
  4      E C A
  5      C D E
  6      D B A
  7      E C D
  8      B C E
  9      C D E
 10      E B C


wandb: uploading artifact run-33x3fkyg-cv_fold_map3_table
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 399-399, summary, console lines 601-618
wandb: 
wandb: Run history:
wandb:                   epoch ▁▃▆█
wandb:                    fold ▁▁▁█
wandb:   fold1/train/epoch_acc ▁▄▇█
wandb:  fold1/train/epoch_loss █▅▂▁
wandb: fold1/train/lr_backbone ▄▅▅▇▇█████▇▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▂▂▂▁▁▁▁▁▁
wandb:     fold1/train/lr_head ▄▄▄▆▇████▇▇▇▆▆▆▅▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
wandb:   fold1/train/step_loss ▆▆▆▆▆▆▅▅▆▅▅▄▆▅▅▆█▅█▄▄▆▄▅▆▃▁▄▆▂▃▃▃▄▂▃▄▃▃▆
wandb:          fold1/val/map3 ▁▆██
wandb:   fold2/train/epoch_acc ▁
wandb:  fold2/train/epoch_loss ▁
wandb:                     +22 ...
wandb: 
wandb: Run summary:
wandb:            cv_map3_mean 0.85817
wandb:             cv_map3_std 0.02711
wandb:                   epoch 4
wandb:                    fold 5
wandb:     fold1/best_val_map3 0.86875
wandb:   fold1/train/epoch_acc 0.63375
wandb:  fold1/train/epoch_loss 1.1239